# 📝 Topic Modeling — หัวข้อหลักของ มอก. บังคับ

**Goal:** ค้นหาหัวข้อ (topics) ที่แฝงอยู่ในข้อความขอบข่ายของ มอก.

**Models:**
1. **NMF** (Non-negative Matrix Factorization) — เหมาะกับ TF-IDF
2. **LDA** (Latent Dirichlet Allocation) — probabilistic topic model

**Output:**
- Top keywords แต่ละ topic
- Topic distribution heatmap
- Topic-document assignment

In [ ]:
# Google Colab — run this cell first
!pip install pandas scikit-learn plotly openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_excel('std-tisi-b stru.xlsx', skiprows=1)
df.columns = ['เลขที่','ชื่อTH','ประกาศ','ชื่อEN','ขอบข่ายTH','ขอบข่ายEN',
              'วันที่ประกาศ','หมวดหมู่','สถานะ','หน่วยงาน','บังคับ']
df['วันที่ประกาศ'] = pd.to_datetime(df['วันที่ประกาศ'], errors='coerce')
df['ปี'] = df['วันที่ประกาศ'].dt.year

# Combine English text
df['text'] = (df['ขอบข่ายEN'].fillna('') + ' ' + df['ชื่อEN'].fillna('')).str.strip()
df = df[df['text'].str.len() > 10].copy()

print(f'Documents: {len(df)}')

## NMF Topic Modeling (TF-IDF based)

In [ ]:
N_TOPICS = 6

# TF-IDF for NMF
tfidf = TfidfVectorizer(max_features=400, stop_words='english',
                        ngram_range=(1,2), min_df=2)
X_tfidf = tfidf.fit_transform(df['text'])
feature_names = tfidf.get_feature_names_out()

nmf = NMF(n_components=N_TOPICS, random_state=42, max_iter=500)
W_nmf = nmf.fit_transform(X_tfidf)  # doc-topic matrix
H_nmf = nmf.components_             # topic-term matrix

print(f'NMF: {N_TOPICS} topics extracted')
print(f'Reconstruction error: {nmf.reconstruction_err_:.4f}')
print()

for i, topic in enumerate(H_nmf):
    top_idx = topic.argsort()[-10:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f'Topic {i}: {" · ".join(top_words)}')

### Chart 1 — NMF Topic-Term Heatmap

In [ ]:
# Get top 8 words per topic
n_words = 8
all_words = set()
topic_word_lists = []
for topic in H_nmf:
    top_idx = topic.argsort()[-n_words:][::-1]
    words = [feature_names[j] for j in top_idx]
    all_words.update(words)
    topic_word_lists.append(words)

all_words = sorted(all_words)
word_idx = {w: i for i, w in enumerate(all_words)}

# Build heatmap matrix
heat_data = np.zeros((N_TOPICS, len(all_words)))
for t, topic in enumerate(H_nmf):
    for w in all_words:
        if w in feature_names.tolist():
            fi = list(feature_names).index(w)
            heat_data[t, word_idx[w]] = topic[fi]

fig_heat = go.Figure(go.Heatmap(
    z=heat_data,
    x=all_words,
    y=[f'Topic {i}' for i in range(N_TOPICS)],
    colorscale='YlOrRd',
    hovertemplate='<b>%{x}</b><br>%{y}<br>Score: %{z:.3f}<extra></extra>',
))
fig_heat.update_layout(
    title='NMF Topic-Term Heatmap',
    xaxis_title='Terms', yaxis_title='Topics',
    template='plotly_white', height=450,
    font=dict(family='Sarabun, sans-serif'),
    xaxis_tickangle=-45,
)
fig_heat.show()

### Chart 2 — Topic Distribution per Document

In [ ]:
df['dominant_topic'] = W_nmf.argmax(axis=1)

topic_dist = df['dominant_topic'].value_counts().sort_index()
PALETTE = ['#003049','#d62828','#f77f00','#2a9d8f','#264653','#e76f51']

fig_dist = go.Figure(go.Bar(
    x=[f'Topic {i}' for i in topic_dist.index],
    y=topic_dist.values,
    marker=dict(color=[PALETTE[i % len(PALETTE)] for i in topic_dist.index]),
    text=topic_dist.values,
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>จำนวน มอก.: %{y}<extra></extra>',
))
fig_dist.update_layout(
    title='จำนวน มอก. ในแต่ละ Topic (NMF)',
    xaxis_title='Topic', yaxis_title='จำนวน มอก.',
    template='plotly_white', height=400,
    font=dict(family='Sarabun, sans-serif'),
)
fig_dist.show()

## LDA Topic Modeling (Count-based)

In [ ]:
# Count vectorizer for LDA
count_vec = CountVectorizer(max_features=400, stop_words='english',
                            ngram_range=(1,2), min_df=2)
X_count = count_vec.fit_transform(df['text'])
count_features = count_vec.get_feature_names_out()

lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42,
                                 max_iter=30, learning_method='online')
W_lda = lda.fit_transform(X_count)

print(f'LDA: {N_TOPICS} topics extracted')
print(f'Log-likelihood: {lda.score(X_count):.2f}')
print(f'Perplexity: {lda.perplexity(X_count):.2f}')
print()

for i, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[-10:][::-1]
    top_words = [count_features[j] for j in top_idx]
    print(f'Topic {i}: {" · ".join(top_words)}')

### Chart 3 — LDA Topic Keywords (Bar Chart)

In [ ]:
fig_lda = go.Figure()
for i, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[-8:][::-1]
    words = [count_features[j] for j in top_idx]
    scores = topic[top_idx] / topic.sum()  # normalize
    
    fig_lda.add_trace(go.Bar(
        x=scores, y=words, orientation='h',
        name=f'Topic {i}',
        marker_color=PALETTE[i % len(PALETTE)],
        visible=(i == 0),  # show first topic by default
    ))

# Add dropdown to select topic
buttons = []
for i in range(N_TOPICS):
    vis = [j == i for j in range(N_TOPICS)]
    buttons.append(dict(label=f'Topic {i}', method='update',
                        args=[{'visible': vis}]))

fig_lda.update_layout(
    title='LDA Topic Keywords (เลือก Topic จาก dropdown)',
    xaxis_title='Normalized Weight',
    template='plotly_white', height=420,
    font=dict(family='Sarabun, sans-serif'),
    margin=dict(l=180),
    updatemenus=[dict(active=0, buttons=buttons,
                      x=0.98, y=1.15, xanchor='right')],
)
fig_lda.show()

### Chart 4 — NMF vs LDA: Topic Agreement

In [ ]:
df['lda_topic'] = W_lda.argmax(axis=1)

agree = pd.crosstab(df['dominant_topic'], df['lda_topic'],
                    rownames=['NMF Topic'], colnames=['LDA Topic'])

fig_agree = go.Figure(go.Heatmap(
    z=agree.values,
    x=[f'LDA {i}' for i in agree.columns],
    y=[f'NMF {i}' for i in agree.index],
    colorscale='Viridis',
    text=agree.values,
    texttemplate='%{text}',
    hovertemplate='NMF %{y} × LDA %{x}<br>จำนวน: %{z}<extra></extra>',
))
fig_agree.update_layout(
    title='NMF vs LDA Topic Agreement',
    template='plotly_white', height=420,
    font=dict(family='Sarabun, sans-serif'),
)
fig_agree.show()

### Chart 5 — Topic × หมวดหมู่จริง

In [ ]:
cross = pd.crosstab(df['dominant_topic'], df['หมวดหมู่'])

fig_cross = go.Figure(go.Heatmap(
    z=cross.values,
    x=cross.columns.tolist(),
    y=[f'Topic {i}' for i in cross.index],
    colorscale='YlOrRd',
    text=cross.values,
    texttemplate='%{text}',
))
fig_cross.update_layout(
    title='NMF Topic × หมวดหมู่จริง',
    xaxis_title='หมวดหมู่', yaxis_title='Topic',
    template='plotly_white', height=420,
    font=dict(family='Sarabun, sans-serif'),
    xaxis_tickangle=-45,
    margin=dict(b=160),
)
fig_cross.show()

## สรุป

Topic Modeling สกัดหัวข้อหลัก 6 หัวข้อจากขอบข่าย มอก. บังคับ 148 รายการ

- **NMF:** เหมาะกับ TF-IDF · ให้ topics ที่ตีความง่าย (sparse)
- **LDA:** probabilistic model · ให้ topic distributions ที่ smooth กว่า
- เปรียบเทียบทั้ง 2 models + ดูความสอดคล้องกับหมวดหมู่จริง